# 10 — Conclusion & Presentation Summary (Paper 1)

**Penguatan Ketangguhan NIDS dengan Reduksi Fitur Berbasis XGBoost Gain-based Importance melalui Adversarial Training terhadap Serangan Evasion Saliency Map**

Notebook ini merangkum output-output kunci dari pipeline notebook 01–09 untuk keperluan **presentasi paper**. Notebook ini **membaca hasil yang sudah ada** (CSV / JSON / PNG) — tidak melatih ulang model — sehingga cepat dijalankan dan aman ditampilkan saat presentasi.

## Alur cerita presentasi
1. **Konteks & Data** — dataset besar & imbalanced, feature importance.
2. **Seleksi Fitur** — penentuan sweet spot Top-10.
3. **Celah Keamanan** — saliency map & degradasi akibat evasion.
4. **Solusi & Hasil** — perbandingan 4 skenario S1–S4, confusion matrix.
5. **Validasi Real-Traffic (AWS)** — offline vs real-traffic.

> Sumber data: `../data/results/*.csv`, `../data/results-nids01/*.json`, dan gambar `../data/*.png`.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import image as mpimg

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

DATA = '../data'
RESULTS = os.path.join(DATA, 'results')
RT = os.path.join(DATA, 'results-nids01')

def show_png(path, title=None, w=9, h=5.5):
    """Tampilkan file PNG hasil notebook sebelumnya jika ada."""
    if os.path.exists(path):
        img = mpimg.imread(path)
        plt.figure(figsize=(w, h))
        plt.imshow(img)
        plt.axis('off')
        if title:
            plt.title(title)
        plt.tight_layout()
        plt.show()
    else:
        print(f'[!] File tidak ditemukan: {path}')

print('Setup selesai. Folder data:', os.path.abspath(DATA))

---
## 1. Konteks & Data

### 1a. Distribusi kelas dataset CSE-CIC-IDS2018 (imbalanced)

In [ ]:
# Distribusi kategori (dari statistik dataset paper). Angka populasi asli.
dist = pd.DataFrame({
    'Kategori': ['Benign', 'DDoS/DoS', 'Brute-Force', 'Botnet', 'Infiltration', 'Web Attacks'],
    'Data Awal': [13484708, 1391082, 380943, 286191, 161934, 928],
})
dist['Subset 10%'] = (dist['Data Awal'] * 0.10).round().astype(int)
dist['Persentase (%)'] = (dist['Data Awal'] / dist['Data Awal'].sum() * 100).round(2)
print('Total sampel:', dist['Data Awal'].sum())
display(dist)

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(dist['Kategori'], dist['Data Awal'], color='#4C72B0')
ax.set_yscale('log')
ax.set_ylabel('Jumlah Sampel (skala log)')
ax.set_title('Distribusi Kategori CSE-CIC-IDS2018 (Class Imbalance Ekstrem)')
for b, v in zip(bars, dist['Data Awal']):
    ax.text(b.get_x()+b.get_width()/2, v, f'{v:,}', ha='center', va='bottom', fontsize=8)
plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

### 1b. Feature Importance (XGBoost Gain) — Top-20

In [ ]:
fi_path = os.path.join(RESULTS, 'feature_importance_all.csv')
fi = pd.read_csv(fi_path)
top20 = fi.head(20).iloc[::-1]  # reverse untuk barh
top10_cum = fi.head(10)['importance'].sum() / fi['importance'].sum() * 100
print(f'Kontribusi Top-10 terhadap total Gain: {top10_cum:.1f}%')

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#C44E52' if i >= 10 else '#55A868' for i in range(len(top20))][::-1]
ax.barh(top20['feature'], top20['importance'], color=colors)
ax.set_xlabel('Importance (Gain)')
ax.set_title('Top-20 Feature Importance (hijau = Top-10 terpilih)')
plt.tight_layout(); plt.show()

---
## 2. Seleksi Fitur: Penentuan Sweet Spot (Top-10)

Perbandingan performa (F1/Precision/Recall) dan efisiensi (ukuran model, inference time) antar konfigurasi fitur. **Data asli** dari `ablation_performance.csv` dan `ablation_efficiency.csv`.

In [ ]:
perf = pd.read_csv(os.path.join(RESULTS, 'ablation_performance.csv'))
eff = pd.read_csv(os.path.join(RESULTS, 'ablation_efficiency.csv'))

# Fokus XGBoost saja untuk presentasi Paper 1
perf_x = perf[perf['Model'] == 'XGBoost'].copy()
eff_x = eff[eff['Model'] == 'XGBoost'].copy()
merged = perf_x.merge(eff_x, on=['Model', 'Features'])
cols = ['Features', 'N_feat', 'Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)',
        'Model Size (MB)', 'Inference/10k (s)']
display(merged[cols])

In [ ]:
# Grafik trade-off: F1 vs jumlah fitur (kiri) + ukuran model (kanan)
m = merged.sort_values('N_feat')
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(m['N_feat'], m['F1-Score (%)'], 'o-', color='#4C72B0', lw=2, label='F1-Score (%)')
ax1.set_xlabel('Jumlah Fitur')
ax1.set_ylabel('F1-Score (%)', color='#4C72B0')
ax1.tick_params(axis='y', labelcolor='#4C72B0')
ax2 = ax1.twinx()
ax2.plot(m['N_feat'], m['Model Size (MB)'], 's--', color='#C44E52', lw=2, label='Ukuran Model (MB)')
ax2.set_ylabel('Ukuran Model (MB)', color='#C44E52')
ax2.tick_params(axis='y', labelcolor='#C44E52')
ax1.axvline(10, color='green', ls=':', alpha=0.7)
ax1.set_title('Trade-off Seleksi Fitur: F1 vs Ukuran Model (garis hijau = Top-10)')
plt.tight_layout(); plt.show()

> **Catatan verifikasi:** ukuran model & F1 di sel di atas diambil langsung dari file hasil (`ablation_efficiency.csv`, `ablation_performance.csv`). Gunakan angka ini sebagai rujukan resmi jika ada perbedaan dengan draf paper.

---
## 3. Celah Keamanan: Saliency Map & Degradasi akibat Evasion

In [ ]:
# Saliency map Top-10 (fitur paling sensitif terhadap perturbasi)
show_png(os.path.join(DATA, 'saliency_map_top10.png'),
         'Saliency Map Top-10: Init Fwd Win Byts paling sensitif')

In [ ]:
# Kurva degradasi MCC & F1 terhadap epsilon (model baseline)
show_png(os.path.join(DATA, 'vulnerability_mcc_f1_vs_epsilon.png'),
         'Degradasi MCC & F1 Model Baseline vs Epsilon Perturbasi')

---
## 4. Solusi & Hasil Utama: Evaluasi 4 Skenario (S1–S4)

> **Sumber terverifikasi:** angka S1–S4 dibaca dari `../data/results/scenarios_s1_s4.csv`. Nilai ini diekstraksi dari grafik hasil `08_evaluation.ipynb` (`evaluation_confusion_2x2.png` untuk MCC 4-desimal & F1, `evaluation_grouped_bar_2x2.png` untuk Precision & Recall).

In [ ]:
# Baca angka S1-S4 terverifikasi dari CSV (diekstrak dari grafik hasil 08_evaluation).
s_path = os.path.join(RESULTS, 'scenarios_s1_s4.csv')
s = pd.read_csv(s_path)
print('Sumber: scenarios_s1_s4.csv (terverifikasi dari evaluation_confusion_2x2.png & evaluation_grouped_bar_2x2.png)')
display(s)

In [ ]:
# Grafik batang 4 skenario (MCC, F1, Precision, Recall)
metrics = ['MCC', 'F1', 'Precision', 'Recall']
palette = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
x = np.arange(len(s)); w = 0.2
fig, ax = plt.subplots(figsize=(10, 5))
for j, (col, c) in enumerate(zip(metrics, palette)):
    offset = (j - (len(metrics)-1)/2) * w
    ax.bar(x + offset, s[col], w, label=col, color=c)
    for i, val in enumerate(s[col]):
        ax.text(i + offset, val + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=6.5, rotation=90)
labels = [f"{r['Skenario']}\n({r['Model']}+{r['Data Uji']})" for _, r in s.iterrows()]
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0, 1.2); ax.set_ylabel('Skor')
ax.set_title('Perbandingan Metrik pada 4 Skenario (S1–S4), \u03b5 = 0,10')
ax.legend(loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.28))
plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrix S2 (baseline+evasion) vs S4 (robust+evasion)
show_png(os.path.join(DATA, 'confusion_s2_vs_s4.png'),
         'Confusion Matrix: S2 (Baseline+Evasion) vs S4 (Robust+Evasion)')

---
## 5. Validasi Real-Traffic (AWS): Offline vs Real

Hasil pengujian model pada trafik jaringan nyata di AWS (RT-S1–RT-S4). **Data asli** dari `../data/results-nids01/RT-S*_metrics.json`.

In [ ]:
# Nilai offline diambil dari CSV terverifikasi (s), dipetakan RT-Sx -> Sx
off_lookup = {f"RT-{row['Skenario']}": row for _, row in s.iterrows()}
rows = []
for sc in ['RT-S1', 'RT-S2', 'RT-S3', 'RT-S4']:
    p = os.path.join(RT, f'{sc}_metrics.json')
    if os.path.exists(p):
        d = json.load(open(p))
        off = off_lookup.get(sc)
        rows.append({'Skenario': sc, 'Model': d.get('model', ''),
                     'MCC_off': off['MCC'], 'F1_off': off['F1'],
                     'MCC_real': d['MCC'], 'Prec_real': d['Precision'],
                     'Recall_real': d['Recall'], 'F1_real': d['F1']})
    else:
        print(f'[!] {p} tidak ditemukan')
rt = pd.DataFrame(rows)
display(rt)

In [ ]:
# Grafik perbandingan MCC offline vs real
if len(rt):
    x = np.arange(len(rt)); w = 0.35
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x - w/2, rt['MCC_off'], w, label='MCC Offline', color='#4C72B0')
    ax.bar(x + w/2, rt['MCC_real'], w, label='MCC Real-Traffic', color='#C44E52')
    ax.axhline(0, color='gray', lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels(rt['Skenario'])
    ax.set_ylabel('MCC'); ax.set_title('MCC: Offline vs Real-Traffic (AWS)')
    ax.legend()
    for i, row in rt.iterrows():
        ax.text(i - w/2, row['MCC_off']+0.02, f"{row['MCC_off']:.2f}", ha='center', fontsize=8)
        ax.text(i + w/2, row['MCC_real']+0.02, f"{row['MCC_real']:.2f}", ha='center', fontsize=8)
    plt.tight_layout(); plt.show()

---
## Ringkasan untuk Presentasi

1. **Data** — CSE-CIC-IDS2018, ~15,7 juta sampel, sangat imbalanced → MCC dipilih sebagai metrik utama.
2. **Seleksi fitur** — Top-10 menyumbang >85% Gain; dipilih sebagai *sweet spot* (performa turun tipis, model jauh lebih ringkas).
3. **Celah keamanan** — reduksi fitur menajamkan gradien; *Init Fwd Win Byts* paling sensitif; baseline kolaps saat evasion (MCC ≈ 0).
4. **Solusi** — *Adversarial Training* memulihkan MCC ke ~0,99 pada data evasion (S4) tanpa merusak akurasi trafik normal (S3).
5. **Validasi real-traffic** — pola ketangguhan konsisten di trafik nyata AWS: baseline rentan, robust bertahan (Precision 1,0). Novelty: validasi nyata, bukan hanya lab.

> **TODO verifikasi:** pastikan angka S1–S4 pada Bagian 4 sesuai output aktual `08_evaluation.ipynb`. Jika `08` diminta menyimpan `scenarios_s1_s4.csv`, notebook ini akan otomatis membacanya.